# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [40]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [41]:
pop_like = cat[cat['product_name'].str.lower().str.contains('pop')]
print(f"{pop_like.shape[0]} rows matched")
pop_like

4 rows matched


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [42]:
pop_like_strawberry = pop_like[pop_like['product_name'].str.lower().str.contains('strawberry')]
pop_not_strawberry = pop_like[~pop_like['product_name'].str.lower().str.contains('strawberry')]

print(pop_like_strawberry)

             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


**Pop-Tart rows that are a different product:** _None of them..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [43]:
CANON = {
    "PT-12": "SKU#459812",
    "POPTART-STRAW": "SKU#459812",
    "PT_STRAWBERRY": "SKU#459812"
}

def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = out.apply(ap, axis=1)
    return out

def ap(row):
    if row['sku'] in CANON:
        return CANON[row['sku']]
    return row['sku']

consolidated = consolidate(cat)
consolidated

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,SKU#459812
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,SKU#459812
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,SKU#459812
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,SKU#459812
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack,WATER24PK
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack,WTR_24PK
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,AA,NaN,3M Company,6 pack,BATT-100
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk,BAT-AA4
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack,BATT_AA_4PK


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [47]:
consolidated_filtered = consolidated.copy()
category_count_before = consolidated.groupby('category').agg(
    count=('category', 'count')
)
print("before")
print(category_count_before)

consolidated_filtered['category'] = consolidated_filtered['category'].str.title()
consolidated_filtered['category'] = consolidated_filtered['category'].str.strip()

category_count_after = consolidated_filtered.groupby('category').agg(
    count=('category', 'count')
)
print("after")
print(category_count_after)

before
                     count
category                  
ALCOHOL                  1
Alcoholic Beverages      1
Baby                     2
Baby Products            1
Breakfast                2
Breakfast Foods          1
ELECTRONICS              2
EMERGENCY                3
ENTERTAINMENT            3
Electronics              1
Emergency                5
Emergency Supplies       3
GROCERY                  3
Groceries                1
Grocery                  1
HOUSEHOLD                1
Household                1
Snacks                   1
baby                     1
breakfast                1
electronics              1
emergency                1
entertainment            2
grocery                  1
health                   2
household                2
snacks                   1
after
                     count
category                  
Alcohol                  1
Alcoholic Beverages      1
Baby                     3
Baby Products            1
Breakfast                3
Breakfast Foods

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [51]:
product_names_normalized = consolidated_filtered.copy()
product_names_normalized['product_name'] = (product_names_normalized['product_name']
                         .str.lower()
                         .str.replace(r'[^a-z ]', '', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (product_names_normalized.groupby('product_name')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

product_name
aa batteries pack                     [BAT-AA4, BATT_AA_4PK]
beer light pack                          [BEER-24, LTBEER24]
board game monopoly        [BG-MONO, MONOPOLY-01, GAME-MONO]
bread white                           [BREAD-WH, WHITEBREAD]
butter peanut                            [PB-16, PNUTBTR-16]
candles emergency                     [CNDL-EM, CANDLE-EMRG]
canned soup                            [SOUP-CAN, CNSOUP-01]
cards playing                       [CARDS-01, PLAYCARD-STD]
chips pack variety                   [CHIPS-VP, CHIPVARIETY]
duty heavy tarp                         [TARP-HD, HDTARP-01]
flashlight led                         [FLASH-01, LED-FLASH]
paper towels                         [PTOWEL-6, PAPERTOWEL6]
poptarts strawberry    [PT-12, POPTART-STRAW, PT_STRAWBERRY]
Name: sku, dtype: object

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [61]:
assert len(cat) == len(product_names_normalized)

assert product_names_normalized['canonical_sku'].notna().all()

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

water bottles seem fragmented by sku and category. put them all in one

whether to mix health and emergency supplies, or to separate them into further categories. ask the store designers